In [1]:
import psycopg2
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
import plotly.express as px

In [2]:
!pip install --upgrade plotly ipywidgets

In [3]:
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [5]:
pio.renderers.default = 'iframe'

In [6]:
sns.set_palette("deep")

host = "***"
port = "***"
dbname = "***"
user = "***"
password = "***"

def fetch_data_pandas(sql_query):
  try:
      connection = psycopg2.connect(
          host=host,
          port=port,
          dbname=dbname,
          user=user,
          password=password
      )

      #print(connection.get_dsn_parameters(), "\n")

      return pd.read_sql(sql_query, connection)

  except Exception as error:
      print("Error while connecting to PostgreSQL", error)

  finally:
      if (connection):
          connection.close()
          print("PostgreSQL connection is closed")


# Выручка по продукту за день

In [7]:
query = """
SELECT
    event_date,
    product_name,
    SUM(revenue) AS daily_revenue
FROM mobile_game.transactions
GROUP BY event_date, product_name
ORDER BY event_date;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [8]:
fig = px.line(df, x='event_date', y='daily_revenue', line_group='product_name', color='product_name')
fig.show()

# Выручка по странам за день

In [47]:
query = """
SELECT
    event_date,
    country,
    SUM(revenue) AS daily_revenue
FROM mobile_game.transactions t
join mobile_game.user_info ui using(user_id)
GROUP BY 1, 2
ORDER BY 1, 2;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [51]:
fig = px.line(df, x='event_date', y='daily_revenue', line_group='country', color='country')
fig.show()

# DAU по каналам

In [52]:
query = """
WITH first_sessions AS (
    SELECT
        user_id,
        MIN(session_start_time) AS first_session_time
    FROM mobile_game.sessions s 
    GROUP BY 1
),
last_click_channel AS (
    SELECT
        fs.user_id,
        ut.channel,
        MAX(ut.touch_date) AS last_touch_date
    FROM first_sessions fs
    LEFT JOIN mobile_game.users_touches ut 
        ON ut.user_id = fs.user_id
        AND ut.touch_date <= fs.first_session_time
    GROUP BY 1, 2
),
last_click_per_user AS (
    SELECT DISTINCT ON (user_id)
        user_id,
        channel,
        last_touch_date
    FROM last_click_channel
    ORDER BY user_id, last_touch_date DESC
)
SELECT
    lcu.last_touch_date,
    lcu.channel,
    COUNT(DISTINCT ui.user_id) AS dau
FROM last_click_per_user lcu
JOIN mobile_game.user_info ui ON ui.user_id = lcu.user_id
GROUP BY 1, 2
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [53]:
fig = px.line(df, x='last_touch_date', y='dau', line_group='channel', color='channel')
fig.show()

# DAU по странам

In [54]:
query = """
WITH first_sessions AS (
    SELECT
        user_id,
        MIN(session_start_time) AS first_session_time
    FROM mobile_game.sessions s 
    GROUP BY 1
),
last_click_channel AS (
    SELECT
        fs.user_id,
        ut.channel,
        MAX(ut.touch_date) AS last_touch_date
    FROM first_sessions fs
    LEFT JOIN mobile_game.users_touches ut 
        ON ut.user_id = fs.user_id
        AND ut.touch_date <= fs.first_session_time
    GROUP BY 1, 2
),
last_click_per_user AS (
    SELECT DISTINCT ON (user_id)
        user_id,
        channel,
        last_touch_date
    FROM last_click_channel
    ORDER BY user_id, last_touch_date DESC
)
SELECT
    lcu.last_touch_date,
    lcu.channel,
    ui.country,
    COUNT(DISTINCT ui.user_id) AS dau
FROM last_click_per_user lcu
JOIN mobile_game.user_info ui ON ui.user_id = lcu.user_id
GROUP BY 1, 2, 3
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [55]:
fig = px.line(df, x='last_touch_date', y='dau', line_group='country', color='country')
fig.show()

# New install по каналам

In [56]:
query = """
WITH new_installs AS (
SELECT
    MIN(user_start_date) AS install_date,
    user_id
FROM mobile_game.user_info
GROUP BY 2
),
last_click_channel AS (
    SELECT
        ni.user_id,
        ut.channel,
        MAX(ut.touch_date) AS last_touch_date
    FROM new_installs ni
    LEFT JOIN mobile_game.users_touches ut 
        ON ut.user_id = ni.user_id
        AND ut.touch_date <= ni.install_date
    GROUP BY 1, 2
),
last_click_per_user AS (
    SELECT DISTINCT ON (user_id)
        user_id,
        channel,
        last_touch_date
    FROM last_click_channel
    ORDER BY user_id, last_touch_date DESC
)
SELECT
    lcu.last_touch_date,
    lcu.channel,
    COUNT(DISTINCT ui.user_id) AS new_installs
FROM last_click_per_user lcu
JOIN mobile_game.user_info ui ON ui.user_id = lcu.user_id
GROUP BY 1, 2
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [57]:
fig = px.line(df, x='last_touch_date', y='new_installs', line_group='channel', color='channel')
fig.show()

# Returning Users

In [58]:
query = """
WITH new_installs AS (
SELECT
    MIN(user_start_date) AS install_date,
    user_id
FROM mobile_game.user_info
GROUP BY 2
),
last_click_channel AS (
    SELECT
        ni.user_id,
        ut.channel,
        MAX(ut.touch_date) AS last_touch_date
    FROM new_installs ni
    LEFT JOIN mobile_game.users_touches ut 
        ON ut.user_id = ni.user_id
        AND ut.touch_date <= ni.install_date
    GROUP BY 1, 2
),
last_click_per_user AS (
    SELECT DISTINCT ON (user_id)
        user_id,
        channel,
        last_touch_date
    FROM last_click_channel
    ORDER BY user_id, last_touch_date DESC
)
SELECT
    lcu.last_touch_date,
    lcu.channel,
    COUNT(s.user_id)::float / count(*) AS returning_1d
FROM last_click_per_user lcu
left JOIN mobile_game.sessions s ON s.user_id = lcu.user_id
and lcu.last_touch_date + 1 = s.session_start_time::date
GROUP BY 1,2
ORDER BY 1;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [59]:
fig = px.line(df, x='last_touch_date', y='returning_1d')
fig.show()

In [60]:
fig = px.line(df, x='last_touch_date', y='returning_1d', line_group='channel', color='channel')
fig.show()

# Daily Conversion по странам

In [61]:
query = """
WITH NewUsers AS (
    SELECT
        DATE(user_start_date) AS install_date,
        COUNT(DISTINCT user_id) AS new_users
    FROM mobile_game.user_info
    GROUP BY 1
),
FirstPurchase AS (
    SELECT
        DATE(ui.user_start_date) AS install_date,
        ui.country,
        COUNT(DISTINCT t.user_id) AS paying_users
    FROM mobile_game.transactions t
    JOIN mobile_game.user_info ui ON t.user_id = ui.user_id
    AND DATE(t.event_date) = DATE(ui.user_start_date)
    GROUP BY 1, 2
)
SELECT
    nu.install_date,
    fp.country,
    CAST(fp.paying_users AS REAL) / nu.new_users AS daily_conversion
FROM NewUsers nu
LEFT JOIN FirstPurchase fp ON nu.install_date = fp.install_date
ORDER BY nu.install_date;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [62]:
fig = px.line(df, x='install_date', y='daily_conversion', line_group='country', color='country')
fig.show()

# ARPPU by Payer Segment and Date

In [63]:
query = """
SELECT
    t.event_date,
    ui.payer_segment,
    SUM(t.revenue) AS total_revenue,
    COUNT(DISTINCT t.user_id) AS paying_users,
    SUM(t.revenue) / COUNT(DISTINCT t.user_id) AS arppu
FROM mobile_game.transactions t
JOIN mobile_game.user_info ui ON t.user_id = ui.user_id
GROUP BY t.event_date, ui.payer_segment
ORDER BY t.event_date, ui.payer_segment;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [64]:
fig = px.line(df, x='event_date', y='arppu', line_group='payer_segment', color='payer_segment')
fig.show()

In [65]:
query = """
WITH RankedTouches AS (
    SELECT
        user_id,
        touch_date,
        channel,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY touch_date DESC) AS rn
    FROM mobile_game.users_touches
),
LastTouch AS (
    SELECT
        user_id,
        touch_date,
        channel
    FROM RankedTouches
    WHERE rn = 1
),
NewUsers AS (
    SELECT
        DATE(ui.user_start_date) AS install_date,
        ui.user_id,
        lt.channel
    FROM mobile_game.user_info ui
    LEFT JOIN LastTouch lt ON ui.user_id = lt.user_id
),
FirstPurchase AS (
    SELECT
        DATE(ui.user_start_date) AS install_date,
        ui.user_id
    FROM mobile_game.transactions t
    JOIN mobile_game.user_info ui ON t.user_id = ui.user_id
    AND DATE(t.event_date) = DATE(ui.user_start_date)
)
SELECT
    nu.install_date,
    nu.channel,
    COUNT(DISTINCT fp.user_id) AS paying_users,
    COUNT(DISTINCT nu.user_id) AS new_users,
    CAST(COUNT(DISTINCT fp.user_id) AS REAL) / COUNT(DISTINCT nu.user_id) AS conversion_rate
FROM NewUsers nu
LEFT JOIN FirstPurchase fp ON nu.user_id = fp.user_id AND DATE(nu.install_date) = DATE(fp.install_date)
GROUP BY nu.install_date, nu.channel
ORDER BY nu.install_date, nu.channel;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [66]:
fig = px.line(df, x='install_date', y='conversion_rate', line_group='channel', color='channel')
fig.show()

In [67]:
query = """
SELECT
    ui.country,
    ui.payer_segment,
    AVG(t.revenue) AS avg_check
FROM mobile_game.transactions t
JOIN mobile_game.user_info ui ON ui.user_id = t.user_id
WHERE t.event_date BETWEEN '2023-01-01' AND '2023-12-31'
GROUP BY ui.country, ui.payer_segment
ORDER BY ui.country, ui.payer_segment;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [68]:
fig = px.line(df, x='payer_segment', y='avg_check', line_group='country', color='country')
fig.show()

In [69]:
query = """
SELECT
    ui.platform,
    ui.payer_segment,
    COUNT(t.transaction_id) AS transactions_count
FROM mobile_game.transactions t
JOIN mobile_game.user_info ui ON ui.user_id = t.user_id
WHERE t.event_date BETWEEN '2023-01-01' AND '2023-12-31'
GROUP BY ui.platform, ui.payer_segment
ORDER BY ui.platform, ui.payer_segment;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [70]:
fig = px.line(df, x='payer_segment', y='transactions_count', line_group='platform', color='platform')
fig.show()

In [71]:
query = """
WITH UserCohorts AS (
    SELECT
        user_id,
        DATE_TRUNC('week', DATE(user_start_date)) AS cohort_week
    FROM mobile_game.user_info
    WHERE country = 'India'
),
WeeklyActiveUsers AS (
    SELECT
        DATE_TRUNC('week', t.event_date) AS week,
        t.user_id
    FROM mobile_game.transactions t
    JOIN mobile_game.user_info ui ON t.user_id = ui.user_id
    WHERE ui.country = 'India'
    GROUP BY 1, 2
)
SELECT
    uc.cohort_week,
    wau.week,
    COUNT(DISTINCT wau.user_id) AS active_users,
    (COUNT(DISTINCT wau.user_id) * 1.0 / (SELECT COUNT(DISTINCT user_id) FROM UserCohorts WHERE cohort_week = uc.cohort_week)) AS retention_rate
FROM UserCohorts uc
LEFT JOIN WeeklyActiveUsers wau ON uc.user_id = wau.user_id
WHERE wau.week >= uc.cohort_week
GROUP BY 1, 2
ORDER BY 1, 2;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [72]:
fig = px.line(df, x='cohort_week', y='retention_rate')
fig.show()

In [75]:
query = """
WITH UserCohorts AS (
    SELECT
        user_id,
        DATE_TRUNC('week', DATE(user_start_date)) AS cohort_week
    FROM mobile_game.user_info
    WHERE country = 'India'
),
WeeklyRevenue AS (
    SELECT
        DATE_TRUNC('week', t.event_date) AS week,
        t.user_id,
        SUM(t.revenue) AS weekly_revenue
    FROM mobile_game.transactions t
    JOIN mobile_game.user_info ui ON t.user_id = ui.user_id
    WHERE ui.country = 'India'
    GROUP BY 1, 2
)
SELECT
    uc.cohort_week,
    wr.week,
    SUM(wr.weekly_revenue) AS total_revenue,
    COUNT(DISTINCT wr.user_id) AS active_users,
    SUM(wr.weekly_revenue) / COUNT(DISTINCT wr.user_id) AS arpu
FROM UserCohorts uc
LEFT JOIN WeeklyRevenue wr ON uc.user_id = wr.user_id
WHERE wr.week >= uc.cohort_week
GROUP BY 1, 2
ORDER BY 1, 2;
"""

df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [77]:
fig = px.line(df, x='cohort_week', y='arpu')
fig.show()

Выполни Лосевский Дмитрий